# Mengapa Harus Dilakukan Dekomposisi (Khususnya Cholesky) pada Gaussian Process?

Dalam formulasi analitik *Gaussian Process Regression* (GPR), kita sering melihat notasi inversi matriks dan determinan:
$$\boldsymbol{\alpha} = K_y^{-1} \mathbf{y} \quad \text{dan} \quad \log |K_y|$$

Secara simbolik di atas kertas, menuliskan $K_y^{-1}$ tampak sangat sederhana. Namun, dalam komputasi nyata (baik implementasi di pustaka seperti PyTorch, GPyTorch, GPflow, Scikit-Learn, maupun kalkulasi numerik manual), **kita hampir tidak pernah menghitung inversi langsung $K_y^{-1}$**. Kita **wajib melakukan dekomposisi**, khususnya **Dekomposisi Cholesky** ($K_y = L L^T$), karena 4 alasan fundamental berikut:

---

## 1. Kestabilan Numerik (*Numerical Stability & Condition Number*)
Matriks kovariansi kernel $K_y$ dibentuk dari fungsi korelasi spasial berbasis jarak (misalnya kernel RBF: $k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2l^2}\right)$).
* Apabila terdapat dua atau lebih titik data training yang lokasinya berdekatan di ruang input, baris dan kolom yang bersesuaian pada matriks $K_y$ akan menjadi sangat mirip (*nearly collinear*).
* Hal ini membuat matriks $K_y$ memiliki **angka kondisi (*condition number*) yang sangat tinggi/buruk** (*ill-conditioned*).
* Jika kita memaksakan inversi langsung (misalnya dengan metode eliminasi Gauss biasa atau Gauss-Jordan), galat pembulatan komputer (*floating-point roundoff error*) akan teramplifikasi secara drastis (*catastrophic numerical explosion*). Akibatnya, inversi bisa gagal atau menghasilkan vektor bobot $\boldsymbol{\alpha}$ yang melenceng jauh dari nilai sebenarnya.
* **Keunggulan Cholesky**: Karena matriks kovariansi $K_y$ bersifat simetris dan positif definit, Dekomposisi Cholesky $K_y = L L^T$ terbukti secara teoretis **sangat stabil tanpa memerlukan pivoting**, sehingga meminimalkan akumulasi galat pembulatan.

---

## 2. Mengubah Inversi Menjadi Substitusi Linier Segitiga yang Cepat & Presisi
Tujuan utama inferensi GPR sebenarnya **bukan mencari matriks inversi $K_y^{-1}$**, melainkan mencari vektor solusi dari sistem persamaan linier:
$$K_y \boldsymbol{\alpha} = \mathbf{y}$$

Dengan mendekomposisi $K_y = L L^T$ di mana $L$ adalah matriks segitiga bawah (*lower triangular matrix*):
$$L \underbrace{(L^T \boldsymbol{\alpha})}_{\mathbf{v}} = \mathbf{y}$$

Masalah inversi yang rumit dipecah menjadi dua tahap substitusi segitiga yang sangat mudah dan presisi:
1. **Forward Substitution** ($L \mathbf{v} = \mathbf{y}$):
   Karena $L$ berbentuk segitiga bawah ($L_{ij} = 0$ untuk $j > i$), elemen pertama langsung diperoleh:
   $$v_1 = \frac{y_1}{L_{11}}$$
   kemudian disubstitusikan ke baris berikutnya untuk mencari $v_2, v_3, \dots, v_N$ secara bertahap.
2. **Backward Substitution** ($L^T \boldsymbol{\alpha} = \mathbf{v}$):
   Karena $L^T$ berbentuk segitiga atas, elemen terakhir langsung diperoleh:
   $$\alpha_N = \frac{v_N}{L_{NN}}$$
   kemudian disubstitusikan mundur ke atas untuk mencari $\alpha_{N-1}, \dots, \alpha_1$.

Kedua tahapan substitusi ini hanya berorde $\mathcal{O}(N^2)$ dan sama sekali tidak melibatkan pembagian determinan matriks besar yang rawan menghasilkan kesalahan numerik.

---

## 3. Menghitung Determinan $\log |K_y|$ Tanpa Risiko *Underflow/Overflow*
Pada tahap optimasi hyperparameter (mengevaluasi *Marginal Likelihood*), kita wajib menghitung determinan matriks kovariansi:
$$\text{Complexity Penalty Term} = -\frac{1}{2} \log |K_y|$$

* Jika $|K_y|$ dihitung secara naif (misalnya lewat ekspansi kofaktor atau perkalian nilai eigen), nilai determinan dari matriks berdimensi sedang hingga besar (misal $N \ge 100$) akan dengan mudah:
  * Menjadi terlampau besar $\to$ mengalami **overflow** (menjadi `+Inf` di memori komputer).
  * Menjadi sangat mendekati nol $\to$ mengalami **underflow** (menjadi `0.0`, yang menyebabkan $\log(0) = -\infty$).
* **Solusi dengan Cholesky**:
  Karena $K_y = L L^T$, determinannya adalah perkalian determinan dari dua matriks segitiga:
  $$|K_y| = |L| \cdot |L^T| = |L|^2$$
  Determinan dari matriks segitiga $L$ hanyalah hasil kali elemen-elemen diagonalnya ($|L| = \prod_{i=1}^N L_{ii}$). Menggunakan sifat logaritma, perkalian raksasa tersebut berubah menjadi **penjumlahan skalar logaritma**:
  $$\log |K_y| = \log \left( \prod_{i=1}^N L_{ii}^2 \right) = 2 \sum_{i=1}^N \log L_{ii}$$
  Operasi ini **100% aman dan kebal terhadap bahaya numerical overflow maupun underflow**.

---

## 4. Efisiensi Komputasi (Menghemat Beban Hitung hingga 50%)
* Inversi matriks umum (seperti eliminasi Gauss atau dekomposisi LU) membutuhkan sekitar **$\frac{2}{3} N^3$ operasi titik kambang (*flops*)**.
* Dekomposisi Cholesky memanfaatkan fakta bahwa $K_y$ simetris ($K_y = K_y^T$). Algoritma hanya perlu menghitung elemen segitiga bawah dan diagonalnya saja, sehingga hanya memerlukan **$\frac{1}{3} N^3$ flops**.
* Artinya, dekomposisi Cholesky **2 kali lebih cepat** dan jauh lebih hemat memori dibandingkan metode inversi umum.

---

## 💡 Rangkuman Jawaban Singkat untuk Dosen Pembimbing
Jika saat bimbingan dosen bertanya: *"Kenapa harus didekomposisi, kenapa tidak langsung di-invers saja matriks kovariansinya?"*, Anda dapat menjawab:
1. **Stabilitas Numerik**: Matriks kovariansi kernel sering kali *ill-conditioned* (baris-barisnya hampir linear dependen akibat titik data yang berdekatan). Inversi langsung akan meledakkan galat pembulatan komputer, sedangkan Cholesky sangat stabil.
2. **Kebutuhan Sistem Linear**: Kita hanya memerlukan vektor bobot $\boldsymbol{\alpha}$, yang jauh lebih akurat dan murah diselesaikan via forward & backward substitution ($L\mathbf{v} = \mathbf{y}$ dan $L^T\boldsymbol{\alpha} = \mathbf{v}$) dibanding mencari inversi eksplisit.
3. **Mencegah Overflow/Underflow pada Determinan**: Determinan $|K_y|$ untuk *marginal likelihood* dapat dihitung secara aman lewat jumlahan logaritma diagonal: $2 \sum \log L_{ii}$.
4. **Efisiensi**: Memangkas kompleksitas komputasi menjadi separuhnya ($\frac{1}{3} N^3$ vs $\frac{2}{3} N^3$).

# Q&A Lanjutan: Apakah Dekomposisi Hanya Karena Menggunakan Komputer? Jika di Atas Kertas, Apakah Bisa Langsung Diinverskan?

### **Pertanyaan:**
> *"Jadi, dekomposisi digunakan karena kita menggunakan komputer untuk menghitung? Apakah jika menghitungnya di atas kertas maka bisa langsung di inverskan saja?"*

### **Jawaban Singkat:**
**Bisa, TAPI hanya praktis untuk ukuran yang sangat kecil ($N = 2$). Begitu ukuran data $N \ge 3$, menghitung langsung invers di atas kertas justru jauh lebih rumit dan melelahkan dibanding dekomposisi.**

---

## 1. Jika Dihitung di Atas Kertas (Aljabar Simbolik Eksak)
Di atas kertas, manusia berpikir dengan angka eksak/pecahan (misal $\frac{1}{3}$, $\sqrt{2}$, atau $e^{-0.5}$).
* Masalah **ketidakstabilan pembulatan numerik (*floating point error*)** tidak terjadi karena kita tidak membulatkan angka ke biner 64-bit seperti CPU komputer.

### Kasus $N = 2$ (Dua Titik Data):
Jika matriksnya $2 \times 2$:
$$K_y = \begin{bmatrix} a & b \\ b & c \end{bmatrix}$$
Di atas kertas, Anda **sangat mudah dan dianjurkan** langsung memakai rumus inversi adjoin-determinan:
$$K_y^{-1} = \frac{1}{ac - b^2} \begin{bmatrix} c & -b \\ -b & a \end{bmatrix}$$
Lalu tinggal kalikan $K_y^{-1} \mathbf{y}$. Pada kasus $2 \times 2$, melakukan Cholesky di atas kertas justru terasa panjang dan bertele-tele.

---

## 2. Mengapa untuk $N \ge 3$ di Atas Kertas pun Dekomposisi Tetap Jauh Lebih Nyaman?
Mari bandingkan apa yang terjadi jika Anda menghitung matriks $3 \times 3$ di atas kertas:

### Cara A: Inversi Langsung via Kofaktor / Adjoin ($K_y^{-1} = \frac{1}{|K_y|} \operatorname{Adj}(K_y)$)
1. Anda harus menghitung **determinan utama $3 \times 3$** (lewat metode Sarrus atau ekspansi baris).
2. Anda harus menghitung **9 determinan kofaktor berukuran $2 \times 2$** satu per satu untuk mengisi tiap elemen matriks adjoin.
3. Seluruh elemen hasil inversi akan berupa angka pecahan rumit dengan penyebut determinan utama.
4. Setelah itu, Anda masih harus melakukan perkalian matriks penuh $3 \times 3$ dengan vektor $\mathbf{y}$ berukuran $3 \times 1$.
$\to$ **Sangat rawan salah hitung tanda minus dan aritmatika kofaktor yang panjang.**

### Cara B: Dekomposisi Cholesky ($K_y = L L^T$)
1. Anda hanya mencari **6 angka** untuk matriks segitiga bawah $L$ (elemen di atas diagonal nol mutlak, tidak perlu dicari).
2. Anda **tidak pernah menghitung 9 kofaktor matriks**.
3. Saat mencari $\boldsymbol{\alpha}$, Anda melakukan **substitusi maju ($L \mathbf{v} = \mathbf{y}$)**:
   * Baris 1 langsung ketemu $v_1 = \frac{y_1}{L_{11}}$ (hanya satu pembagian biasa).
   * Baris 2 langsung ketemu $v_2$.
   * Baris 3 langsung ketemu $v_3$.
4. Lalu mundur di $L^T \boldsymbol{\alpha} = \mathbf{v}$ untuk mendapatkan $\alpha_3, \alpha_2, \alpha_1$.
$\to$ **Jauh lebih sedikit langkah tulisannya, lebih terstruktur, dan mudah ditiru di papan tulis.**

---

## 3. Kesimpulan Dua Perspektif

| Aspek Peninjauan | Mengapa Tidak Inversi Langsung? | Mengapa Memilih Dekomposisi? |
|---|---|---|
| **Perspektif Komputer (Numerik)** | **Kestabilan angka**: Inversi langsung pada matriks kernel yang *ill-conditioned* memicu ledakan galat pembulatan desimal (*roundoff explosion*) dan *overflow/underflow* pada determinan. | Algoritma Cholesky stabil tanpa pivoting, cepat, dan aman dari *overflow* via $2 \sum \log L_{ii}$. |
| **Perspektif Kertas (Manusia)** | **Beban aljabar**: Untuk $N \ge 3$, mencari matriks inversi eksplisit mengharuskan kita menghitung banyak sekali kofaktor minor yang sangat melelahkan. | Kita tidak butuh matriks inversnya, hanya butuh solusinya. Substitusi segitiga maju-mundur jauh lebih mudah dikerjakan manual. |

> **Fakta Sejarah**: Tokoh matematika **André-Louis Cholesky** menemukan metode dekomposisi ini pada awal abad ke-20 untuk keperluan survei topografi militer—**jauh sebelum komputer digital modern diciptakan**—justru karena ia merasa memecahkan sistem persamaan linier di atas kertas dengan faktorisasi segitiga jauh lebih cepat dan minim kesalahan dibandingkan menginvers matriks secara manual!

# Penjelasan Lengkap Aljabar: Bagaimana Dekomposisi Cholesky Menghasilkan Parameter Distribusi Normal Posterior?

Di dalam teori *Gaussian Process Regression* (GPR), distribusi posterior untuk titik uji $X_*$ berbentuk distribusi normal multivariat:
$$\mathbf{f}_* \mid X, \mathbf{y}, X_* \sim \mathcal{N}\big(\bar{\mathbf{f}}_*, \operatorname{cov}(\mathbf{f}_*)\big)$$

Dua parameter yang wajib kita hitung secara aljabar adalah:
1. **Parameter Mean Posterior**: $\bar{\mathbf{f}}_* = K(X_*, X) K_y^{-1} \mathbf{y} \in \mathbb{R}^{N_* \times 1}$
2. **Parameter Kovariansi/Variansi Posterior**: $\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - K(X_*, X) K_y^{-1} K(X, X_*) \in \mathbb{R}^{N_* \times N_*}$

Keduanya sama-sama memuat suku inversi $K_y^{-1}$. Mari kita bedah langkah aljabar penurunan dekomposisi $K_y = L L^T$ untuk menghitung kedua parameter tersebut tanpa pernah mencari $K_y^{-1}$ secara eksplisit.

---

## 1. Aljabar Penurunan Parameter Mean Posterior ($\bar{\mathbf{f}}_*$)

Rumus analitik mean adalah:
$$\bar{\mathbf{f}}_* = K(X_*, X) \underbrace{\left[ K_y^{-1} \mathbf{y} \right]}_{\boldsymbol{\alpha}}$$

### Langkah A: Definisikan Vektor Bobot $\boldsymbol{\alpha}$
Kita ingin mencari vektor $\boldsymbol{\alpha} \in \mathbb{R}^{N \times 1}$ sedemikian sehingga:
$$\boldsymbol{\alpha} = K_y^{-1} \mathbf{y} \iff K_y \boldsymbol{\alpha} = \mathbf{y}$$

### Langkah B: Substitusikan Faktorisasi Cholesky $K_y = L L^T$
$$(L L^T) \boldsymbol{\alpha} = \mathbf{y} \implies L \left( L^T \boldsymbol{\alpha} \right) = \mathbf{y}$$

### Langkah C: Pecah Menjadi Dua Sistem Linier Segitiga
Definisikan vektor perantara $\mathbf{v} = L^T \boldsymbol{\alpha} \in \mathbb{R}^{N \times 1}$. Persamaan menjadi sistem berantai:
$$\begin{cases}
1.\quad L \mathbf{v} = \mathbf{y} & \text{(Sistem Segitiga Bawah / Forward Substitution)} \\[0.8em]
2.\quad L^T \boldsymbol{\alpha} = \mathbf{v} & \text{(Sistem Segitiga Atas / Backward Substitution)}
\end{cases}$$

Secara aljabar elemen per elemen:
* **Menyelesaikan $L \mathbf{v} = \mathbf{y}$ (Maju dari $i = 1$ hingga $N$):**
  $$\begin{bmatrix} 
  L_{11} & 0 & \dots & 0 \\ 
  L_{21} & L_{22} & \dots & 0 \\ 
  \vdots & \vdots & \ddots & \vdots \\ 
  L_{N1} & L_{N2} & \dots & L_{NN} 
  \end{bmatrix} 
  \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \end{bmatrix} 
  = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_N \end{bmatrix}$$
  * Baris 1: $L_{11} v_1 = y_1 \implies v_1 = \frac{y_1}{L_{11}}$
  * Baris 2: $L_{21} v_1 + L_{22} v_2 = y_2 \implies v_2 = \frac{y_2 - L_{21} v_1}{L_{22}}$
  * Rumus umum rekursif: $$v_i = \frac{1}{L_{ii}} \left( y_i - \sum_{k=1}^{i-1} L_{ik} v_k \right)$$

* **Menyelesaikan $L^T \boldsymbol{\alpha} = \mathbf{v}$ (Mundur dari $i = N$ hingga $1$):**
  $$\begin{bmatrix} 
  L_{11} & L_{21} & \dots & L_{N1} \\ 
  0 & L_{22} & \dots & L_{N2} \\ 
  \vdots & \vdots & \ddots & \vdots \\ 
  0 & 0 & \dots & L_{NN} 
  \end{bmatrix} 
  \begin{bmatrix} \alpha_1 \\ \alpha_2 \\ \vdots \\ \alpha_N \end{bmatrix} 
  = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \end{bmatrix}$$
  * Baris $N$: $L_{NN} \alpha_N = v_N \implies \alpha_N = \frac{v_N}{L_{NN}}$
  * Baris $N-1$: $L_{N-1,N-1} \alpha_{N-1} + L_{N, N-1} \alpha_N = v_{N-1} \implies \alpha_{N-1} = \frac{v_{N-1} - L_{N, N-1} \alpha_N}{L_{N-1, N-1}}$
  * Rumus umum mundur: $$\alpha_i = \frac{1}{L_{ii}} \left( v_i - \sum_{k=i+1}^N L_{ki} \alpha_k \right)$$

### Langkah D: Hitung Mean Prediktif Akhir
Setelah vektor $\boldsymbol{\alpha}$ diperoleh utuh, kalikan dengan matriks kovariansi silang $K(X_*, X)$:
$$\bar{\mathbf{f}}_* = K(X_*, X) \boldsymbol{\alpha} = \sum_{i=1}^N \alpha_i K(X_*, \mathbf{x}_i)$$
*(Selesai! Parameter mean posterior diperoleh murni lewat eliminasi aljabar biasa).* 

---

## 2. Aljabar Penurunan Parameter Kovariansi Posterior ($\operatorname{cov}(\mathbf{f}_*)$)

Rumus analitik kovariansi posterior adalah:
$$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - \underbrace{K(X_*, X) K_y^{-1} K(X, X_*)}_{\Omega}$$
Kita perlu menghitung suku pengurang $\Omega \in \mathbb{R}^{N_* \times N_*}$ secara cerdas menggunakan dekomposisi Cholesky $K_y = L L^T$.

### Langkah A: Manipulasi Aljabar Menggunakan Sifat Transpose & Invers
Ingat sifat dasar aljabar invers perkalian matriks:
$$(A B)^{-1} = B^{-1} A^{-1} \implies (L L^T)^{-1} = (L^T)^{-1} L^{-1} = L^{-T} L^{-1}$$

Substitusikan ke dalam suku $\Omega$:
$$\Omega = K(X_*, X) \left[ L^{-T} L^{-1} \right] K(X, X_*)$$

Ingat bahwa matriks kovariansi silang bersifat transpose: $K(X_*, X) = K(X, X_*)^T$. Maka:
$$\Omega = K(X, X_*)^T L^{-T} L^{-1} K(X, X_*)$$

Dengan sifat aljabar transpose $(A B)^T = B^T A^T$, perhatikan suku:
$$K(X, X_*)^T L^{-T} = \left( L^{-1} K(X, X_*) \right)^T$$

### Langkah B: Definisikan Matriks Perantara $W$
Definisikan matriks $W \in \mathbb{R}^{N \times N_*}$ sebagai:
$$W = L^{-1} K(X, X_*)$$

Maka secara otomatis:
$$W^T = \left( L^{-1} K(X, X_*) \right)^T = K(X, X_*)^T L^{-T} = K(X_*, X) L^{-T}$$

Dengan demikian, suku pengurang $\Omega$ yang tadinya sangat rumit **berubah menjadi bentuk perkalian matriks simetris yang sangat elegan**:
$$\Omega = W^T W$$

### Langkah C: Cara Mencari Matriks $W$ Kolom Demi Kolom Tanpa Menginvers $L$
*(Bagian ini yang menjadi fokus pertanyaan Anda: Bagaimana dari $W = L^{-1} K(X, X_*)$ bisa berubah menjadi $L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$?)*

Mari kita bedah struktur internal matriks $W$ dan matriks $K(X, X_*)$ kolom per kolom!

1. **Tinjau Struktur Matriks $K(X, X_*)$ Berukuran $N \times N_*$**:
   Matriks kovariansi silang ini menghubungkan $N$ data training $X = [\mathbf{x}_1, \dots, \mathbf{x}_N]^T$ dengan $N_*$ titik uji $X_* = [\mathbf{x}_{*1}, \dots, \mathbf{x}_{*N_*}]^T$.
   Matriks ini tersusun atas kolom-kolom vektor:
   $$K(X, X_*) = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix} \in \mathbb{R}^{N \times N_*}$$
   di mana kolom ke-$j$ adalah vektor kovariansi titik uji ke-$j$ terhadap seluruh data training:
   $$\mathbf{k}_{*j} = K(X, \mathbf{x}_{*j}) = \begin{bmatrix} k(\mathbf{x}_1, \mathbf{x}_{*j}) \\ k(\mathbf{x}_2, \mathbf{x}_{*j}) \\ \vdots \\ k(\mathbf{x}_N, \mathbf{x}_{*j}) \end{bmatrix} \in \mathbb{R}^{N \times 1}$$

2. **Tinjau Struktur Matriks $W$ Berukuran $N \times N_*$**:
   Sama halnya dengan $K(X, X_*)$, matriks $W$ juga tersusun atas $N_*$ buah vektor kolom:
   $$W = \begin{bmatrix} \mathbf{w}_1 & \mathbf{w}_2 & \dots & \mathbf{w}_{N_*} \end{bmatrix} \in \mathbb{R}^{N \times N_*}$$
   di mana masing-masing kolom $\mathbf{w}_j = [w_{1j}, w_{2j}, \dots, w_{Nj}]^T \in \mathbb{R}^{N \times 1}$.

3. **Kalikan Persamaan $W = L^{-1} K(X, X_*)$ dengan $L$ dari Kiri**:
   $$L W = L \left( L^{-1} K(X, X_*) \right) \implies L W = K(X, X_*)$$

4. **Tuliskan dalam Bentuk Blok Kolom**:
   Sesuai sifat dasar aljabar linier, perkalian matriks dengan matriks terbagi menjadi perkalian matriks dengan masing-masing kolomnya:
   $$L \begin{bmatrix} \mathbf{w}_1 & \mathbf{w}_2 & \dots & \mathbf{w}_{N_*} \end{bmatrix} = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix}$$
   $$\begin{bmatrix} L \mathbf{w}_1 & L \mathbf{w}_2 & \dots & L \mathbf{w}_{N_*} \end{bmatrix} = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix}$$

5. **Pemisahan Sistem Linier per Kolom**:
   Dengan menyamakan kolom ke-$j$ di sisi kiri dan sisi kanan, kita peroleh sistem persamaan linier independen untuk tiap titik uji ke-$j$ ($j = 1, \dots, N_*$):
   $$L \mathbf{w}_j = \mathbf{k}_{*j} \iff L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$$

   **Mengapa bentuk ini sangat menguntungkan?**
   Karena $L$ adalah matriks segitiga bawah (*lower triangular*), persamaan $L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$ diselesaikan dengan **Forward Substitution biasa** dari baris 1 sampai baris $N$:
   $$\begin{bmatrix} 
   L_{11} & 0 & 0 \\ 
   L_{21} & L_{22} & 0 \\ 
   L_{31} & L_{32} & L_{33} 
   \end{bmatrix} 
   \begin{bmatrix} w_{1j} \\ w_{2j} \\ w_{3j} \end{bmatrix}
   = \begin{bmatrix} k(\mathbf{x}_1, \mathbf{x}_{*j}) \\ k(\mathbf{x}_2, \mathbf{x}_{*j}) \\ k(\mathbf{x}_3, \mathbf{x}_{*j}) \end{bmatrix}$$
   * $w_{1j} = \frac{k(\mathbf{x}_1, \mathbf{x}_{*j})}{L_{11}}$
   * $w_{2j} = \frac{k(\mathbf{x}_2, \mathbf{x}_{*j}) - L_{21} w_{1j}}{L_{22}}$
   * $w_{3j} = \frac{k(\mathbf{x}_3, \mathbf{x}_{*j}) - L_{31} w_{1j} - L_{32} w_{2j}}{L_{33}}$
   *(Sangat mudah dan cepat, persis sama seperti mencari $\mathbf{v}$ pada perhitungan mean sebelumnya!).*

### Langkah D: Hitung Kovariansi & Variansi Posterior Akhir
Setelah semua kolom $\mathbf{w}_j$ ditemukan sehingga matriks $W = [\mathbf{w}_1, \dots, \mathbf{w}_{N_*}]$ lengkap:
$$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - W^T W$$

Perhatikan elemen matriks $W^T W$:
* Elemen diagonal ke-$j$ (variansi titik uji ke-$j$):
  $$[W^T W]_{jj} = \mathbf{w}_j^T \mathbf{w}_j = \|\mathbf{w}_j\|^2 = \sum_{i=1}^N w_{ij}^2$$
  Maka **variansi marjinal prediktif** pada titik $\mathbf{x}_{*j}$ adalah:
  $$\mathbb{V}[f_{*j}] = k(\mathbf{x}_{*j}, \mathbf{x}_{*j}) - \|\mathbf{w}_j\|^2$$

* Elemen non-diagonal baris $j$ kolom $k$ (kovariansi silang antara titik uji $\mathbf{x}_{*j}$ dan $\mathbf{x}_{*k}$):
  $$[W^T W]_{jk} = \mathbf{w}_j^T \mathbf{w}_k = \sum_{i=1}^N w_{ij} w_{ik}$$
  Maka **kovariansi posterior** antara dua titik uji adalah:
  $$\operatorname{cov}(f_{*j}, f_{*k}) = k(\mathbf{x}_{*j}, \mathbf{x}_{*k}) - \mathbf{w}_j^T \mathbf{w}_k$$

---

## 3. Rangkuman Peta Alur Aljabar Cholesky pada GPR

```
      Matriks Kovariansi Ky = K(X, X) + sigma_n^2 * I
                             │
                             ▼
                [ Faktorisasi Cholesky ]
                       Ky = L * L^T
                             │
             ┌───────────────┴───────────────┐
             ▼                               ▼
   [ MENCARI MEAN f_* ]            [ MENCARI KOVARIANSI cov(f_*) ]
             │                               │
   1. Selesaikan: L * v = y        1. Selesaikan tiap kolom j:
      (Forward substitution)          L * w_j = K(X, x_*j)
             │                        (Forward substitution)
             │                               │
   2. Selesaikan: L^T * alpha = v  2. Hitung suku pengurang:
      (Backward substitution)         Omega = W^T * W
             │                               │
   3. Kalikan:                     3. Kurangkan dari Prior:
      f_* = K(X_*, X) * alpha         cov(f_*) = K(X_*, X_*) - W^T * W
```

### Keindahan Aljabar Ini:
1. **Tidak ada satu pun proses inversi matriks** ($K_y^{-1}$ hilang digantikan oleh sifat $(L L^T)^{-1} = L^{-T} L^{-1}$). 
2. Variansi posterior $\mathbb{V}[f_{*j}] = k_{**} - \|\mathbf{w}_j\|^2$ secara aljabar menjamin nilai variansi **selalu berkurang atau sama dengan prior** ($k_{**}$ dikurangi kuadrat norma $\|\mathbf{w}_j\|^2 \ge 0$). Ini mencerminkan hukum probabilitas: *informasi data training tidak pernah menambah ketidakpastian prior*.

# Q&A: Mengapa Hanya Blok Kovariansi $K(X, X)$ yang Memiliki Komponen Galat?

Pertanyaan mendasar: Pada matriks kovariansi gabungan (*joint Gaussian prior*):
$$\begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix},
\begin{bmatrix}
K_y & K(X, X_*) \\
K(X_*, X) & K(X_*, X_*)
\end{bmatrix}
\right)$$
mengapa hanya blok kiri-atas yang bernilai $K_y = K(X,X) + \sigma_n^2 I_N$, sedangkan tiga blok lainnya ($K(X, X_*)$, $K(X_*, X)$, dan $K(X_*, X_*)$) tidak memiliki suku galat $\sigma_n^2 I$?

---

## 1. Meninjau Definisi Variabel yang Dipasangkan
Matriks kovariansi gabungan sebenarnya adalah kovariansi berpasangan antara vektor observasi training $\mathbf{y}$ dan vektor fungsi laten test $\mathbf{f}_*$:

$$\operatorname{cov}\left( \begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \right) = 
\begin{bmatrix}
\operatorname{cov}(\mathbf{y}, \mathbf{y}) & \operatorname{cov}(\mathbf{y}, \mathbf{f}_*) \\
\operatorname{cov}(\mathbf{f}_*, \mathbf{y}) & \operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*)
\end{bmatrix}$$

Ingat model observasinya:
* Data training observasi: $\mathbf{y} = \mathbf{f} + \boldsymbol{\epsilon}$, di mana $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \sigma_n^2 I_N)$.
* Nilai fungsi test yang ingin diprediksi: $\mathbf{f}_* = f(X_*)$ (secara sengaja dipilih sebagai **fungsi murni / laten**, tanpa galat).

---

## 2. Bedah Aljabar 4 Blok Kovariansi

### A. Blok Kiri-Atas: $\operatorname{cov}(\mathbf{y}, \mathbf{y}) = K(X, X) + \sigma_n^2 I_N$
Kedua variabel yang dipasangkan adalah data training yang tercemar galat pengukuran:
$$\begin{aligned}
\operatorname{cov}(\mathbf{y}, \mathbf{y}) &= \operatorname{cov}(\mathbf{f} + \boldsymbol{\epsilon}, \; \mathbf{f} + \boldsymbol{\epsilon}) \\
&= \operatorname{cov}(\mathbf{f}, \mathbf{f}) + \underbrace{\operatorname{cov}(\mathbf{f}, \boldsymbol{\epsilon})}_{0} + \underbrace{\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f})}_{0} + \operatorname{cov}(\boldsymbol{\epsilon}, \boldsymbol{\epsilon}) \\
&= K(X, X) + \sigma_n^2 I_N
\end{aligned}$$
$\implies$ Karena $\boldsymbol{\epsilon}$ bertemu dengan sesamanya, muncul variansi galat $\sigma_n^2 I_N$.

---

### B. Blok Kanan-Bawah: $\operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*) = K(X_*, X_*)$
Perhatikan notasinya: kita menggunakan $\mathbf{f}_*$, bukan $\mathbf{y}_*$.
* $\mathbf{f}_*$ adalah **fungsi laten murni** pada test points, sehingga secara matematis **noise-free**:
  $$\mathbf{f}_* = f(X_*)$$
* Karena tidak ada komponen galat $\boldsymbol{\epsilon}_*$ pada $\mathbf{f}_*$, maka kovariansinya murni berasal dari fungsi kernel:
  $$\operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*) = K(X_*, X_*)$$

*(Catatan: Jika tujuan Anda adalah memprediksi nilai pengukuran sensor baru $\mathbf{y}_* = \mathbf{f}_* + \boldsymbol{\epsilon}_*$, barulah blok ini bernilai $K(X_*, X_*) + \sigma_n^2 I$).*

---

### C. Blok Silang: $\operatorname{cov}(\mathbf{y}, \mathbf{f}_*) = K(X, X_*)$
Blok ini mengukur kovariansi silang antara observasi training dengan fungsi laten test:
$$\begin{aligned}
\operatorname{cov}(\mathbf{y}, \mathbf{f}_*) &= \operatorname{cov}(\mathbf{f} + \boldsymbol{\epsilon}, \; \mathbf{f}_*) \\
&= \operatorname{cov}(\mathbf{f}, \mathbf{f}_*) + \operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*)
\end{aligned}$$
* $\operatorname{cov}(\mathbf{f}, \mathbf{f}_*) = K(X, X_*)$ (korelasi antar fungsi murni melalui fungsi kernel).
* $\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*) = 0$, karena **galat instrumen pengukuran masa lalu ($\boldsymbol{\epsilon}$) tidak mungkin berkorelasi dengan fungsi murni masa depan ($\mathbf{f}_*$)**. Keduanya saling bebas (*independent*).

Sehingga suku galat lenyap, menyisakan $K(X, X_*)$. Begitu pula untuk transpose-nya $K(X_*, X) = K(X, X_*)^T$.

---

## 3. Rangkuman Inti untuk Jawaban ke Dosen
1. **Target inferensi kita adalah fungsi murni ($\mathbf{f}_*$)**: Kita ingin merekonstruksi kurva asli yang bersih dari gangguan alat ukur (*noise-free reconstruction*). Oleh karena itu, $\mathbf{f}_*$ tidak mengandung $\boldsymbol{\epsilon}$.
2. **Independensi galat**: Galat pengukuran training $\boldsymbol{\epsilon}$ bersifat independen terhadap nilai fungsi murni di mana pun, sehingga kovariansi silangnya bernilai nol ($\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*) = 0$).
3. **Akibatnya**: Suku $\sigma_n^2 I_N$ hanya hidup di blok $\operatorname{cov}(\mathbf{y}, \mathbf{y}) = K_y$.

# Q&A: Apa Kegunaan Subbab 4.2 (Teorema Partisi Gaussian Bersyarat / Schur Complement)?

Mengapa Subbab 4.2 perlu ada di dalam dokumen dan tidak langsung saja ke rumus prediksi di Subbab 4.3?

---

## 1. Inti Masalah yang Dihadapi
Di Subbab 4.1, kita baru memiliki **distribusi bersama (*Joint Gaussian Prior*)** berukuran $(N + N_*)$:
$$\begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix},
\begin{bmatrix}
K_y & K(X, X_*) \\
K(X_*, X) & K(X_*, X_*)
\end{bmatrix}
\right)$$

Ini adalah distribusi gabungan sebelum kita memproses data. Padahal tujuan utama Machine Learning / Regresi adalah **melakukan prediksi bersyarat**:
$$\text{"Jika kita sudah mengamati data latih } \mathbf{y}\text{, bagaimanakah peluang nilai fungsi di titik uji } \mathbf{f}_*\text{?"}$$
yaitu mencari distribusi bersyarat $p(\mathbf{f}_* \mid \mathbf{y})$.

---

## 2. Peran Subbab 4.2: Menyediakan Teorema Baku Statistik
Subbab 4.2 menyediakan rumus umum aljabar linear dan probabilitas Gaussian multivariat.

Jika dua kelompok variabel acak sembarang $\mathbf{z}_A$ dan $\mathbf{z}_B$ berdistribusi normal bersama:
$$\begin{bmatrix} \mathbf{z}_A \\ \mathbf{z}_B \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \boldsymbol{\mu}_A \\ \boldsymbol{\mu}_B \end{bmatrix},
\begin{bmatrix} \Sigma_{AA} & \Sigma_{AB} \\ \Sigma_{BA} & \Sigma_{BB} \end{bmatrix}
\right)$$
maka distribusi bersyarat $\mathbf{z}_B$ jika $\mathbf{z}_A$ diketahui juga **pasti berdistribusi normal secara eksak**:
$$\mathbf{z}_B \mid \mathbf{z}_A \sim \mathcal{N}(\boldsymbol{\mu}_{B|A}, \Sigma_{B|A})$$
dengan rumus baku:
$$\begin{aligned}
\boldsymbol{\mu}_{B|A} &= \boldsymbol{\mu}_B + \Sigma_{BA} \Sigma_{AA}^{-1} (\mathbf{z}_A - \boldsymbol{\mu}_A) \\
\Sigma_{B|A} &= \Sigma_{BB} - \Sigma_{BA} \Sigma_{AA}^{-1} \Sigma_{AB}
\end{aligned}$$

---

## 3. Kegunaan di Subbab 4.3: Tinggal Mencocokkan Variabel (*Plug & Play*)
Dengan adanya Subbab 4.2, penurunan rumus di Subbab 4.3 menjadi sangat elegan dan runtut tanpa perlu menurunkan integral dari nol:

Kita tinggal memetakan variabel dokumen ke teorema Subbab 4.2:
* $\mathbf{z}_A \leftarrow \mathbf{y}$ (data training) dengan mean $\boldsymbol{\mu}_A = \mathbf{0}$
* $\mathbf{z}_B \leftarrow \mathbf{f}_*$ (fungsi laten test) dengan mean $\boldsymbol{\mu}_B = \mathbf{0}$
* $\Sigma_{AA} \leftarrow K_y = K(X,X) + \sigma_n^2 I_N$
* $\Sigma_{BA} \leftarrow K(X_*, X)$
* $\Sigma_{AB} \leftarrow K(X, X_*)$
* $\Sigma_{BB} \leftarrow K(X_*, X_*)$

Substitusikan ke rumus Subbab 4.2:
1. **Mean Prediksi Posterior:**
   $$\bar{\mathbf{f}}_* = \mathbf{0} + K(X_*, X) K_y^{-1} (\mathbf{y} - \mathbf{0}) = K(X_*, X) K_y^{-1} \mathbf{y}$$
2. **Kovariansi Posterior:**
   $$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - K(X_*, X) K_y^{-1} K(X, X_*)$$

Kedua persamaan di atas adalah **jantung persamaan dari Gaussian Process Regression**. Tanpa Subbab 4.2, kedua rumus ini akan terkesan "jatuh dari langit".

---

## 4. Makna Fisis Schur Complement (Tanda Minus $- $)
Suku $\Sigma_{BB} - \Sigma_{BA}\Sigma_{AA}^{-1}\Sigma_{AB}$ disebut sebagai **Schur Complement** dari blok $\Sigma_{AA}$.

Makna fisis dari tanda minus ($-$) tersebut sangat intuitif:
* $K(X_*, X_*)$ adalah **ketidakpastian awal (*prior uncertainty*)** di titik uji sebelum kita punya data apa pun.
* Suku $K(X_*, X) K_y^{-1} K(X, X_*)$ bernilai selalu semi-definit positif.
* Oleh karena itu, pengurangan ini menunjukkan bahwa **adanya data observasi $\mathbf{y}$ selalu MENGURANGI ketidakpastian** di titik uji. Semakin dekat titik uji ke data latih, suku pengurang semakin besar, sehingga variansi posteriornya mendekati nol (sangat yakin).

---

## 5. Ringkasan Singkat untuk Dosen
> *"Subbab 4.2 berfungsi sebagai landasan teoretis fundamental (teorema conditioning Gaussian multivariat) yang menjembatani distribusi gabungan prior di Subbab 4.1 menuju formulasi analitik posterior di Subbab 4.3. Melalui Schur Complement pada Subbab 4.2, penurunan rumus mean dan variansi prediksi GPR dapat dibuktikan secara matematis dan runtut."*

# Q&A: Mengapa Kita Harus Memperhatikan Keterdiferensialan (*Differentiability*) pada Gaussian Process?

### **Pertanyaan Inti:**
> *"Mengapa sifat keterdiferensialan (khususnya mean-square differentiability dari fungsi kernel) sangat penting diperhatikan dalam Gaussian Process? Apa dampak pemilihan kelas keterdiferensialan terhadap pemodelan matematika, komputasi numerik, dan aplikasi segmentasi citra?"*

---

## 1. Definisi Matematis: Hubungan Kernel dengan Keterdiferensialan Sampel Fungsi

Dalam *Gaussian Process* (GP), sifat kehalusan dan keterdiferensialan dari realisasi sampel fungsi $f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}'))$ **ditentukan sepenuhnya oleh keterdiferensialan fungsi kernel $k(\mathbf{x}, \mathbf{x}')$ di titik asal ($r = \|\mathbf{x} - \mathbf{x}'\| = 0$)**.

### Teorema *Mean-Square Differentiability* (Rasmussen & Williams, 2006, Bab 4):
Suatu proses stokastik $f(\mathbf{x})$ dikatakan **terdiferensiasi $k$-kali secara *mean-square*** jika dan hanya jika turunan parsial ke-$2k$ dari fungsi kovariansinya ada dan kontinu pada $\mathbf{x} = \mathbf{x}'$:
$$\exists \left. \frac{\partial^{2k} k(\mathbf{x}, \mathbf{x}')}{\partial x_i^k \partial x_j'^k} \right|_{\mathbf{x}=\mathbf{x}'} < \infty$$

### Spektrum Keterdiferensialan pada Kernel Populer:
1. **Exponential / Ornstein-Uhlenbeck ($k(r) = \sigma_f^2 \exp(-r/l)$)**:
   * Turunan satu sisi di $r=0$: $\left. \frac{dk}{dr} \right|_{r=0^+} = -\frac{\sigma_f^2}{l} \neq 0$ (memiliki sudut tajam / *cusp*).
   * **Kelas Kehalusan**: $C^0$ (Hanya kontinu, **tidak terdiferensiasi sama sekali** secara *mean-square*).
   * **Bentuk Sampel**: Sangat kasar dan bergerigi (*jagged*), identik dengan Gerak Brown / Wiener Process 1D.

2. **Matérn $\nu = 3/2$ ($k_{3/2}(r) = \sigma_f^2 (1 + \frac{\sqrt{3}r}{l}) \exp(-\frac{\sqrt{3}r}{l})$)**:
   * **Kelas Kehalusan**: $C^1$ (**1 kali terdiferensiasi** secara *mean-square*).
   * **Bentuk Sampel**: Memiliki gradien pertama yang kontinu, tetapi turunan kedua (kelengkungan/akselerasi) tidak kontinu.

3. **Matérn $\nu = 5/2$ ($k_{5/2}(r) = \sigma_f^2 (1 + \frac{\sqrt{5}r}{l} + \frac{5r^2}{3l^2}) \exp(-\frac{\sqrt{5}r}{l})$)**:
   * **Kelas Kehalusan**: $C^2$ (**2 kali terdiferensiasi** secara *mean-square*).
   * **Bentuk Sampel**: Memiliki kurvatur kontinu, tampak mulus bagi mata manusia namun tetap mempertahankan variasi lokal yang realistis.

4. **Squared Exponential / RBF ($k_{\text{SE}}(r) = \sigma_f^2 \exp(-\frac{r^2}{2l^2})$)**:
   * Turunan di $r=0$: $\left. \frac{dk}{dr} \right|_{r=0^+} = 0$, dan turunan tingkat tingginya ada hingga tak terhingga.
   * **Kelas Kehalusan**: $C^\infty$ (**Terdiferensiasi tak hingga kali** secara *mean-square* / fungsi analitik).
   * **Bentuk Sampel**: Sangat mulus (*infinitely smooth*).

---

## 2. Empat Alasan Utama Mengapa Keterdiferensialan Wajib Diperhatikan

---

### A. Realisme Fisik Data & Menghindari *Over-smoothing* (Kritik Stein)
* Banyak praktisi pemula memilih kernel *Squared Exponential* (RBF) hanya karena formulanya sederhana. Padahal, asumsi keterdiferensialan tak hingga ($C^\infty$) **sangat tidak realistis untuk sebagian besar data empiris dunia nyata**.
* **Kritik Ahli Statistik Michael L. Stein (1999)** (*Interpolation of Spatial Data: Some Theory for Kriging*):
  > *"Asumsi kehalusan $C^\infty$ pada Squared Exponential terlalu kuat dan jarang dijumpai pada proses fisik nyata. Mengetahui fungsi di suatu selang kecil akan membuat kita dapat memprediksi seluruh fungsi beserta seluruh ordenya secara deterministik."*
* **Dampaknya pada Pemodelan**:
  * Jika data memiliki fluktuasi lokal, sudut tajam, atau turbulensi, memaksakan kernel $C^\infty$ akan menyebabkan **over-smoothing** (detail lokal hilang dan kurva diratakan secara paksa).
  * Menggunakan keluarga **Matérn ($\nu=3/2$ atau $\nu=5/2$)** memberikan tingkat kehalusan yang jauh lebih realistis untuk fenomena fisik dan spasial.

---

### B. Kestabilan Komputasi Numerik Matriks Kovariansi ($K$)
* Sifat keterdiferensialan kernel berhubungan langsung dengan **kondisi numerik (*condition number*) matriks kovariansi $K$**.
* Pada kernel $C^\infty$ (seperti RBF), titik-titik data yang posisinya saling berdekatan memiliki korelasi yang luar biasa tinggi dan meluruh sangat lambat di sekitar $r=0$.
  * Hal ini menyebabkan baris-baris pada matriks $K$ menjadi hampir identik (*nearly collinear*).
  * Matriks $K$ menjadi **sangat buruk kondisinya (*ill-conditioned*)**, dengan nilai eigen terkecil mendekati nol atau bahkan negatif karena galat pembulatan floating-point (IEEE 754).
  * Akibatnya, **Dekomposisi Cholesky ($K = L L^T$) rentan gagal numerik**, sehingga memaksa kita menambahkan *jitter* regularisasi diagonal ($\epsilon_{\text{jitter}} I$) yang lebih besar.
* Sebaliknya, kernel dengan keterdiferensialan lebih rendah (seperti Matérn $3/2$ atau $5/2$) memiliki spektrum nilai eigen yang meluruh lebih lambat (sesuai ruang Sobolev), sehingga matriks kovariansinya **jauh lebih stabil secara numerik (*well-conditioned*)**.

---

### C. Fleksibilitas Ekstrapolasi & Estimasi Ketidakpastian (*Uncertainty*)
* Derajat keterdiferensialan mengontrol **seberapa kaku fungsi mengunci arah pergerakannya**:
  * **Kernel $C^\infty$ (RBF)**: Karena semua turunan ordenya kontinu dan mulus, fungsi dipaksa sangat kaku. Ketika melakukan ekstrapolasi ke luar wilayah titik data observasi, ketidakpastian prediktif seringkali menjadi kurang realistis karena korelasi spasial terlalu memaksakan kelengkungan global.
  * **Kernel Matérn ($C^1, C^2$)**: Memungkinkan fungsi berbelok lebih fleksibel. Estimasi ketidakpastian (*predictive variance*) membesar secara lebih wajar dan proporsional saat menjauhi titik observasi, mencerminkan ketidaktahuan model secara jujur.

---

### D. Relevansi Krusial pada Segmentasi Citra & Deep Gaussian Process (DGP)
* **Karakteristik Citra pada Tugas Akhir**:
  * Citra digital tersusun atas wilayah homogen (*background/foreground*) yang dipisahkan oleh **tepi objek (*edges / contours*)**.
  * Tepi objek merupakan bentuk **diskontinuitas atau perubahan intensitas yang sangat tajam (gradien spasial tinggi/tak hingga)**.
* **Mengapa Single GP dengan Kernel $C^\infty$ Gagal pada Segmentasi Citra?**
  * Asumsi stasioner dan kehalusan $C^\infty$ dari kernel RBF memaksakan tingkat kehalusan yang seragam di seluruh piksel citra.
  * Akibatnya, batas tepi segmentasi menjadi kabur (*blurry/over-smoothed boundaries*). Jika *lengthscale* $l$ diperkecil untuk menangkap tepi tajam, model akan kehilangan kemampuan generalisasi pada area yang halus dan matriks menjadi *ill-conditioned*.
* **Jembatan Menuju Deep Gaussian Process (DGP)**:
  * Inilah motivasi utama beralih ke arsitektur hirarkis **Deep Gaussian Process**: $\mathbf{y} = f_L(f_{L-1}(\dots f_1(\mathbf{x})))$.
  * Lapisan-lapisan GP laten melakukan pemetaan koordinat (*spatial warping*), membengkokkan ruang input sedemikian rupa sehingga area tepi objek yang kasar dapat dimodelkan secara adaptif tanpa merusak kehalusan di area lainnya.

---

## 3. Matriks Perbandingan Spektrum Keterdiferensialan Kernel

| Nama Kernel | Formula $k(r)$ | Kelas Kehalusan | Karakteristik Sampel Fungsi | Kestabilan Numerik Matriks $K$ | Rekomendasi Kasus Penggunaan |
|---|---|---|---|---|---|
| **Exponential (Matérn $\nu=1/2$)** | $\sigma_f^2 \exp\left(-\frac{r}{l}\right)$ | $C^0$ (Tidak terdiferensiasi) | Sangat kasar, bergerigi tajam (*Brownian path*) | Sangat stabil (*well-conditioned*) | Proses stokastik acak murni, data finansial/harga saham berfluktuasi tinggi |
| **Matérn $\nu=3/2$** | $\sigma_f^2 \left(1 + \frac{\sqrt{3}r}{l}\right) \exp\left(-\frac{\sqrt{3}r}{l}\right)$ | $C^1$ (Sekali terdiferensiasi) | Kontinu dengan sudut halus, gradien kontinu | Stabil | Data spasial fisik, geostatistika, permukaan topografi alami, data citra |
| **Matérn $\nu=5/2$** | $\sigma_f^2 \left(1 + \frac{\sqrt{5}r}{l} + \frac{5r^2}{3l^2}\right) \exp\left(-\frac{\sqrt{5}r}{l}\right)$ | $C^2$ (Dua kali terdiferensiasi) | Mulus secara visual, kelengkungan/akselerasi kontinu | Cukup stabil | Pemodelan teknik mesin, dinamika fluida, regresi permukaan halus, segmentasi citra |
| **Squared Exponential (RBF)** | $\sigma_f^2 \exp\left(-\frac{r^2}{2l^2}\right)$ | $C^\infty$ (Terdiferensiasi tak hingga) | Luar biasa mulus, tidak ada sudut tajam sama sekali | Kurang stabil (*ill-conditioned*, rawan gagal Cholesky jika data rapat) | Fungsi analitik murni, medan potensial gravitasi/elektrostatik ideal |

---

## 💡 Rangkuman Jawaban Singkat untuk Dosen Pembimbing
Jika saat bimbingan atau ujian dosen bertanya: *"Kenapa kita harus memperhatikan keterdiferensialan fungsi kernel pada GP?"*, Anda dapat menjawab secara sistematis dengan 4 poin:
1. **Representasi Realistis (Inductive Bias)**: Keterdiferensialan kernel menentukan kehalusan fungsi sampel. Kernel $C^\infty$ (RBF) mengasumsikan dunia ini terlampau mulus sehingga memicu *over-smoothing*, sedangkan kernel Matérn ($C^1$ atau $C^2$) memodelkan fenomena alami secara jauh lebih realistis (sebagaimana kritik Michael Stein).
2. **Kestabilan Numerik**: Semakin tinggi keterdiferensialan ($C^\infty$), matriks kovariansi semakin *ill-conditioned* dan rawan gagal pada dekomposisi Cholesky. Kernel Matérn menghasilkan matriks yang jauh lebih stabil.
3. **Kualitas Ekstrapolasi**: Menentukan fleksibilitas kurva dalam memperkirakan ketidakpastian (*uncertainty*) di luar wilayah data observasi.
4. **Konteks Segmentasi Citra & Deep GP**: Citra memiliki batas tepi objek (*edges*) yang tidak mulus (gradien tajam). Memahami batas keterdiferensialan kernel tunggal adalah alasan mendasar mengapa kita memerlukan **Deep Gaussian Process (DGP)** untuk melakukan *spatial warping* non-stasioner pada batas objek segmentasi.

# Analisis & Rekomendasi Perombakan Section 1 Dokumen Kredit #3: Membahas Seluruh Kernel Buku Teks & Eksperimen Parameternya

### **Pertanyaan Diskusi:**
> *"Untuk section pertama dari `kredit_bimbingan_03`, saya ingin merombak strukturnya. Saya ingin kita membahas semua kernel yang ada di textbook dan melakukan eksperimen terhadap nilai parameternya. Bagaimana menurutmu? Jika ada yang kurang dan perlu ditambahkan silahkan beritahu saya."*

---

## 1. Analisis Kritis & Opini Objektif

### A. Sisi Positif (Kelebihan Pendekatan Ini):
1. **Pemahaman Taksonomi Ruang Fungsi yang Utuh**: Membahas seluruh spektrum kernel (stasioner, non-stasioner, periodik, multiskala, hingga koneksi *neural network*) membuat draf landasan teori skripsi Anda memiliki kedalaman akademis yang sangat kuat setara buku teks standar (*Rasmussen & Williams, 2006; Duvenaud, 2014*).
2. **Kekayaan Visual & Eksperimental**: Melakukan simulasi eksperimen pada parameter setiap kernel (misal $\alpha$ pada *Rational Quadratic*, $p$ pada *Periodic*, $\nu$ pada *Matérn*, serta ARD) akan menghasilkan grafik-grafik komparasi berkualitas publikasi yang membuktikan intuisi matematis secara nyata.
3. **Membuka Wawasan Rekayasa Kernel (*Kernel Engineering*)**: Mahasiswa mampu menunjukkan bagaimana operasi aljabar kernel (penjumlahan dan perkalian) dapat mengombinasikan sifat-sifat dasar (seperti tren linier + variasi periodik + derau putih).

---

### B. Peringatan Kritis & Risiko (Berdasarkan Catatan Bimbingan #2):
Ingat kembali arahan dan teguran dosen pembimbing pada **Catatan Bimbingan #2 (16 September 2026)**:
> *\"Dosen menilai materi yang terlalu umum/melebar hanya berstatus 'good to know', bukan fokus utama. Dokumen kredit sejatinya disiapkan sebagai **draf bab landasan teori laporan Tugas Akhir (skripsi)**. Isi kredit harus berfokus langsung pada konsep-konsep yang menjadi fondasi langsung menuju **Deep Gaussian Process** dan **Image Segmentation**.\"*

**Potensi Bahaya**:
Jika kita membahas seluruh kernel sekadar sebagai "daftar katalog / ensiklopedia" (seperti merangkum kamus kernel dari A sampai Z tanpa benang merah yang jelas), dosen dapat kembali mengkritik:
* *\"Kenapa kamu membahas kernel periodik atau polinomial begitu panjang lebar, padahal topik skripsimu adalah Segmentasi Citra dengan Deep GP?\"*

---

### C. Solusi Strategis: Pendekatan *\"Goal-Oriented Kernel Taxonomy\"*
Agar rencana perombakan ini **diterima dengan sangat baik dan dipuji oleh dosen**, kita tidak boleh menyajikannya sebagai katalog acak. Kita harus menyusunnya dengan **struktur taksonomi berbasis tujuan (*goal-oriented*) yang bermuara langsung pada pemodelan spasial citra dan motivasi arsitektur Deep GP**.

---

## 2. Taksonomi Lengkap Kernel Buku Teks (Rasmussen & Williams Bab 4 + Duvenaud Cookbook)

Berikut klasifikasi lengkap kernel standar yang dapat dimasukkan ke dalam dokumen:

```
                                  SPEKTRUM FUNGSI KERNEL
                                             │
         ┌───────────────────────────────────┴───────────────────────────────────┐
         ▼                                                                       ▼
  [ KERNEL STASIONER ]                                                [ KERNEL NON-STASIONER ]
  (Invarian translasi: k(x, x') = k(x - x'))                          (Tergantung lokasi absolut: k(x, x'))
         │                                                                       │
 ┌───────┴────────────────────────┐                                      ┌───────┴────────────────────────┐
 ▼                                ▼                                      ▼                                ▼
[ ISOTROPIK (Berbasis Jarak r) ]  [ PERIODIK ]                     [ DOT PRODUCT / POLINOMIAL ]   [ NEURAL NETWORK / SIGMOID ]
- Squared Exponential (RBF)       - Exp-Sine-Squared               - Linear Kernel                - Williams (1998) Arc-sine
- Exponential (Ornstein-Uhl.)     - Cosine Kernel                  - Polynomial Kernel            - Neal (1995) Infinite NN
- Matérn (nu = 1/2, 3/2, 5/2)                                                                     - Gibbs Non-stationary
- Rational Quadratic (Multiscale)
- White Noise Kernel
```

### Ringkasan Formula Matematis Kernel Buku Teks:

1. **Kernel Stasioner & Isotropik**:
   * **Squared Exponential / RBF**: $k_{\text{SE}}(r) = \sigma_f^2 \exp\left(-\frac{r^2}{2l^2}\right)$ $\to$ Kelas $C^\infty$, fungsi sangat mulus.
   * **Exponential (Matérn 1/2)**: $k_{\text{Exp}}(r) = \sigma_f^2 \exp\left(-\frac{r}{l}\right)$ $\to$ Kelas $C^0$, kasar bergerigi (*Brownian motion*).
   * **Matérn 3/2 & 5/2**: $k_{3/2}(r) = \sigma_f^2 \left(1 + \frac{\sqrt{3}r}{l}\right) \exp\left(-\frac{\sqrt{3}r}{l}\right)$ dan $k_{5/2}(r) = \sigma_f^2 \left(1 + \frac{\sqrt{5}r}{l} + \frac{5r^2}{3l^2}\right) \exp\left(-\frac{\sqrt{5}r}{l}\right)$ $\to$ Kelas $C^1$ dan $C^2$, paling realistis untuk data fisik.
   * **Rational Quadratic (RQ)**: $k_{\text{RQ}}(r) = \sigma_f^2 \left(1 + \frac{r^2}{2\alpha l^2}\right)^{-\alpha}$ $\to$ Merupakan **superposisi/campuran tak hingga dari kernel SE dengan berbagai lengthscale** yang berdistribusi Gamma ($p(l^{-2}) \sim \text{Gamma}(\alpha, \alpha l_0^2)$). Sangat bagus untuk data multiskala.
   * **White Noise Kernel**: $k_{\text{WN}}(\mathbf{x}, \mathbf{x}') = \sigma_n^2 \delta(\mathbf{x} - \mathbf{x}')$ $\to$ Memodelkan ketidakpastian independen antar titik observasi.

2. **Kernel Periodik (Stasioner Khusus)**:
   * **Exp-Sine-Squared / Periodic**: $k_{\text{Per}}(r) = \sigma_f^2 \exp\left(-\frac{2\sin^2(\pi r / p)}{l^2}\right)$ $\to$ Parameter $p$ mengatur panjang periode pengulangan, $l$ mengatur kehalusan variasi dalam satu siklus.
   * **Cosine Kernel**: $k_{\text{Cos}}(r) = \sigma_f^2 \cos\left(\frac{2\pi r}{p}\right)$.

3. **Kernel Non-Stasioner (Dot-Product & Deep Connection)**:
   * **Linear Kernel**: $k_{\text{Lin}}(\mathbf{x}, \mathbf{x}') = \sigma_b^2 + \sigma_v^2 (\mathbf{x} - \mathbf{c})^T (\mathbf{x}' - \mathbf{c})$ $\to$ Setara dengan Bayesian Linear Regression. Variansi membesar seiring menjauh dari pusat $\mathbf{c}$.
   * **Polynomial Kernel**: $k_{\text{Poly}}(\mathbf{x}, \mathbf{x}') = (\sigma_0^2 + \mathbf{x}^T \mathbf{x}')^d$.
   * **Neural Network (Arc-sine / Erf) Kernel (Williams, 1998)**:
     $$k_{\text{NN}}(\mathbf{x}, \mathbf{x}') = \frac{2}{\pi} \arcsin \left( \frac{2 \tilde{\mathbf{x}}^T \Sigma \tilde{\mathbf{x}}'}{\sqrt{(1 + 2 \tilde{\mathbf{x}}^T \Sigma \tilde{\mathbf{x}})(1 + 2 \tilde{\mathbf{x}}'^T \Sigma \tilde{\mathbf{x}}')}} \right)$$
     di mana $\tilde{\mathbf{x}} = [1, \mathbf{x}^T]^T$. **Sangat penting!** Menjadi bukti matematis bahwa Single-Layer Neural Network dengan jumlah neuron tak terhingga konvergen ke Gaussian Process (Neal, 1995).

4. **Aljabar & Komposisi Kernel**:
   * **Penjumlahan ($k_1 + k_2$)**: Menggabungkan struktur secara aditif (misal: Tren Linier + Musiman Periodik + Variasi Lokal Matérn + Derau Putih).
   * **Perkalian ($k_1 \times k_2$)**: Memodulasi sifat secara lokal (misal: *Locally Periodic* $k_{\text{Per}} \times k_{\text{SE}}$, di mana pola periodik dapat meluruh/berubah seiring jarak).
   * **Automatic Relevance Determination (ARD)**: Menggunakan matriks diagonal $M = \operatorname{diag}(l_1^{-2}, \dots, l_D^{-2})$ sehingga tiap dimensi input (misal sumbu X dan Y pada piksel citra) memiliki skala korelasi independen.

---

## 3. Usulan Struktur Baru Section 1 pada Dokumen Kredit #3

Berikut rancangan struktur Section 1 yang rapi, komprehensif, dan memiliki narasi yang kuat menuju Tugas Akhir:

* **1.1 Fondasi Matematis & Syarat Keabsahan Kernel (PSD, Mercer, Bochner)**
* **1.2 Spektrum Keterdiferensialan & Kehalusan Spasial (Exponential, Matérn 1/2, 3/2, 5/2, Squared Exponential)**
  * Menjelaskan turunan satu sisi pada $r=0$ dan konsep *mean-square differentiability*.
* **1.3 Pemodelan Multiskala & Struktur Khusus (Rational Quadratic, Periodic, dan White Noise)**
  * Pembuktian analitik bahwa Rational Quadratic adalah campuran kontinu kernel RBF multiskala dengan parameter $\alpha$.
* **1.4 Kernel Non-Stasioner & Jembatan Teoretis ke Deep Learning (Linear, Polynomial, Neural Network Kernel)**
  * Mengulas Teorema Radford Neal (1995) & Christopher Williams (1998) tentang hubungan GP dan Jaringan Saraf Tiruan.
* **1.5 Aljabar Komposisi Kernel & Automatic Relevance Determination (ARD)**
  * Aturan operasi penutupan (*closure properties*): penjumlahan, perkalian, dan konvolusi.
* **1.6 Desain Eksperimen & Analisis Komparasi Parameter (1D Signal & 2D Image Space)**
  * Menampilkan galeri visualisasi profil kernel, sampel fungsi 1D, dan peta kovariansi spasial piksel 2D.
* **1.7 Keterbatasan Fundamental Single-Layer GP untuk Segmentasi Citra (Motivasi Menuju Deep GP)**
  * Kesimpulan kritis: Mengapa merekayasa kernel tunggal (meskipun dengan aljabar rumit) tetap tidak cukup untuk menangani batas diskontinuitas objek citra yang non-stasioner $\to$ Justifikasi kebutuhan Deep GP.

---

## 4. Rencana Eksperimen Parameter yang Perlu Dilakukan

Untuk menyertai pembahasan teori di atas, berikut eksperimen komputasi yang diusulkan untuk dibuat visualisasinya:

| Eksperimen | Parameter yang Divariasikan | Visualisasi yang Dihasilkan | Pesan Ilmiah yang Disampaikan |
|---|---|---|---|
| **Eksperimen 1: Spektrum Kehalusan** | Kernel: Exp ($C^0$), Matérn 3/2 ($C^1$), Matérn 5/2 ($C^2$), RBF ($C^\infty$) | 4-Panel Sampel Fungsi 1D | Menunjukkan transisi dari kasar bergerigi hingga mulus analitik. |
| **Eksperimen 2: Parameter Skala & Amplitudo** | $l \in \{0.2, 1.0, 3.0\}$ dan $\sigma_f^2 \in \{0.25, 1.0, 4.0\}$ | Grafik Sampel Multi-kurva | $l$ mengatur frekuensi horizontal, $\sigma_f^2$ mengatur rentang vertikal. |
| **Eksperimen 3: Multiskala Rational Quadratic** | $\alpha \in \{0.1, 0.5, 2.0, \infty\}$ pada $k_{\text{RQ}}$ | Kurva Kernel $k(r)$ & Sampel 1D | Saat $\alpha \to \infty$, RQ konvergen tepat ke RBF; saat $\alpha$ kecil, RQ menangkap variasi dari berbagai rentang jarak sekaligus. |
| **Eksperimen 4: Struktur Periodik & Modulasi Lokal** | Periode $p \in \{1.0, 3.0\}$ dan Komposit $k_{\text{Per}} \times k_{\text{SE}}$ | Sampel Fungsi Periodik Murni vs *Locally Periodic* | Menunjukkan bagaimana perkalian kernel membatasi periodisitas hanya pada wilayah lokal. |
| **Eksperimen 5: Kernel Non-Stasioner (Linear & NN)** | Slope $\sigma_v^2$, Offset $c$, dan Weight Variance $\Sigma$ pada NN Kernel | Sampel Fungsi Non-Stasioner | Memperlihatkan bahwa variansi fungsi berubah secara dinamis tergantung posisi $x$. |
| **Eksperimen 6: Domain Spasial 2D (Piksel Citra) & ARD** | $l_x = l_y$ (Isotropik) vs $l_x \neq l_y$ (Anisotropik ARD) | Kontur 2D & Permukaan 3D $f(x_1, x_2)$ | Relevansi langsung ke koordinat piksel citra 2D tugas akhir Anda. |

---

## 5. Rekomendasi Hal yang Perlu Ditambahkan (*Missing Elements / Value-Add*)

Agar draf ini sempurna untuk persiapan tugas akhir Anda, ada **3 hal krusial yang wajib ditambahkan**:

1. **Visualisasi Domain 2D ($D=2$)**:
   * Tugas akhir Anda adalah *Image Segmentation*. Input citra adalah grid koordinat 2D $(x_1, x_2)$ atau fitur intensitas warna.
   * Jika seluruh eksperimen hanya disajikan dalam 1D, relevansinya dengan citra kurang terasa. Menambahkan plot kontur kovariansi 2D dan visualisasi fungsi sampel 2D (berupa citra tekstur sintetik) akan membuat dosen sangat terkesan.
2. **Koneksi Teoretis GP $\leftrightarrow$ Neural Network (Neal, 1995)**:
   * Menjelaskan mengapa GP disebut sebagai generalisasi non-parametrik dari Jaringan Saraf Tiruan dengan lebar lapisan tersembunyi tak terhingga (*infinite-width limit*). Ini menjadi jembatan konseptual yang sangat elegan menuju *Deep Gaussian Process*.
3. **Analisis Kestabilan Numerik Komparatif (Condition Number)**:
   * Menyertakan grafik perbandingan *condition number* matriks $K$ untuk masing-masing kernel terhadap kerapatan data. Ini membuktikan secara empiris mengapa kernel Matérn jauh lebih stabil daripada RBF saat data sangat padat.

---

## 💡 Kesimpulan: Apakah Ide Ini Direkomendasikan?
**Sangat Direkomendasikan**, dengan catatan:
* Format penyajiannya harus mengikuti **taksonomi berbasis tujuan (*goal-oriented*)**, bukan sekadar salinan katalog buku.
* Setiap pembahasan kernel harus diakhiri dengan evaluasi: *bagaimana karakteristik kernel tersebut bila diterapkan pada domain spasial citra?*
* Bagian akhir Section 1 harus menjadi batu loncatan yang menegaskan bahwa *kombinasi kernel stasioner pada Single GP tidak cukup untuk segmentasi batas objek citra yang non-stasioner*, sehingga mutlak diperlukan **Deep Gaussian Process**.

# Panduan Lengkap: Katalog Kernel Rasmussen & Williams (Bab 4) dan Eksperimen Variasi Hyperparameter

Sesuai arahan, fokus Section 1 disederhanakan: **tidak membahas penurunan analitik kehalusan/turunan**, melainkan berfokus penuh pada **katalog fungsi kernel standar dari buku teks Rasmussen & Williams (2006, Bab 4)** dan **analisis komprehensif efek variasi nilai hyperparameternya**.

---

## 1. Katalog Lengkap Fungsi Kernel Buku Teks Rasmussen & Williams (2006)

Berdasarkan *Gaussian Processes for Machine Learning* (Rasmussen & Williams, 2006, Tabel 4.1 & Bab 4), seluruh fungsi kovariansi dikelompokkan ke dalam dua kategori utama:

### A. Kernel Stasioner & Isotropik (Hanya Bergantung pada Jarak Skalar $r = \|\mathbf{x} - \mathbf{x}'\|$)

| No | Nama Kernel | Formula Matematis $k(r)$ | Hyperparameter | Peran / Deskripsi Fisis Hyperparameter |
|---|---|---|---|---|
| 1 | **Squared Exponential (SE / RBF / Gaussian)** | $k_{\text{SE}}(r) = \sigma_f^2 \exp\left(-\frac{r^2}{2l^2}\right)$ | - $l > 0$ (*Lengthscale*)<br>- $\sigma_f^2 > 0$ (*Signal Variance*) | - $l$: Mengatur rentang jarak korelasi spasial horizontal.<br>- $\sigma_f^2$: Mengatur skala variansi/amplitudo vertikal fungsi. |
| 2 | **Keluarga Matérn ($\nu = 1/2, 3/2, 5/2$)** | $k_{\text{Matérn}}(r) = \sigma_f^2 \frac{2^{1-\nu}}{\Gamma(\nu)} \left(\frac{\sqrt{2\nu}r}{l}\right)^\nu K_\nu\left(\frac{\sqrt{2\nu}r}{l}\right)$<br><br>*Bentuk eksplisit:*<br>- $\nu=1/2$: $\sigma_f^2 \exp(-r/l)$ (*Exponential*)<br>- $\nu=3/2$: $\sigma_f^2 (1 + \frac{\sqrt{3}r}{l}) \exp(-\frac{\sqrt{3}r}{l})$<br>- $\nu=5/2$: $\sigma_f^2 (1 + \frac{\sqrt{5}r}{l} + \frac{5r^2}{3l^2}) \exp(-\frac{\sqrt{5}r}{l})$ | - $l > 0$<br>- $\sigma_f^2 > 0$<br>- $\nu > 0$ (*Shape / Smoothness*) | - $\nu$: Mengontrol kekasaran fluktuasi lokal sampel fungsi (dari kasar $\nu=1/2$ hingga mulus $\nu=5/2$). |
| 3 | **Rational Quadratic (RQ)** | $k_{\text{RQ}}(r) = \sigma_f^2 \left(1 + \frac{r^2}{2\alpha l^2}\right)^{-\alpha}$ | - $l > 0$<br>- $\sigma_f^2 > 0$<br>- $\alpha > 0$ (*Scale Mixture*) | - $\alpha$: Mengatur bobot campuran multi-skala. Merupakan superposisi kontinu kernel SE dengan lengthscale berbeda-beda. Saat $\alpha \to \infty$, RQ menjadi kernel SE biasa. |
| 4 | **$\gamma$-Exponential** | $k_{\gamma}(r) = \sigma_f^2 \exp\left(-\left(\frac{r}{l}\right)^\gamma\right), \quad 0 < \gamma \le 2$ | - $l > 0$<br>- $\sigma_f^2 > 0$<br>- $\gamma \in (0, 2]$ (*Power*) | - $\gamma$: Mengontrol bentuk kelengkungan di dekat titik asal. Kasus $\gamma=1$ adalah Exponential, $\gamma=2$ adalah Squared Exponential. |
| 5 | **Periodic (Exp-Sine-Squared)** | $k_{\text{Per}}(r) = \sigma_f^2 \exp\left(-\frac{2\sin^2(\pi r / p)}{l^2}\right)$ | - $p > 0$ (*Period*)<br>- $l > 0$ (*Lengthscale*)<br>- $\sigma_f^2 > 0$ | - $p$: Menentukan jarak perulangan siklus gelombang periodik.<br>- $l$: Menentukan elastisitas/kehalusan osilasi di dalam satu periode. |
| 6 | **Cosine** | $k_{\text{Cos}}(r) = \sigma_f^2 \cos\left(\frac{2\pi r}{p}\right)$ | - $p > 0$ (*Period*)<br>- $\sigma_f^2 > 0$ | - $p$: Periode perulangan gelombang kosinus murni. |
| 7 | **White Noise (Independent Noise)** | $k_{\text{WN}}(\mathbf{x}, \mathbf{x}') = \sigma_n^2 \delta(\mathbf{x} - \mathbf{x}')$ | - $\sigma_n^2 > 0$ (*Noise Variance*) | - $\sigma_n^2$: Tingkat ketidakpastian/derau independen pada tiap pengukuran. |

---

### B. Kernel Non-Stasioner (Tergantung pada Koordinat Absolut $\mathbf{x}$ dan $\mathbf{x}'$)

| No | Nama Kernel | Formula Matematis $k(\mathbf{x}, \mathbf{x}')$ | Hyperparameter | Peran / Deskripsi Fisis Hyperparameter |
|---|---|---|---|---|
| 8 | **Linear (Dot Product)** | $k_{\text{Lin}}(\mathbf{x}, \mathbf{x}') = \sigma_b^2 + \sigma_v^2 (\mathbf{x} - c)(\mathbf{x}' - c)$ | - $\sigma_b^2 \ge 0$ (*Bias Variance*)<br>- $\sigma_v^2 > 0$ (*Slope Variance*)<br>- $c \in \mathbb{R}$ (*Center/Offset*) | - $c$: Titik tumpu (*pivot*) di mana variansi minimum bernilai $\sigma_b^2$.<br>- $\sigma_v^2$: Kecepatan melebarnya variansi linier menjauhi titik $c$. |
| 9 | **Polynomial** | $k_{\text{Poly}}(\mathbf{x}, \mathbf{x}') = \left(\sigma_0^2 + \sigma_v^2 \mathbf{x}^T \mathbf{x}'\right)^d$ | - $\sigma_0^2 \ge 0$ (*Bias/Inhomogeneity*)<br>- $\sigma_v^2 > 0$ (*Scale*)<br>- $d \in \mathbb{N}$ (*Degree*) | - $d$: Derajat kurva polinomial ($d=2$ kuadratik, $d=3$ kubik, dst.).<br>- $\sigma_0^2$: Mengatur interaksi orde rendah. |
| 10 | **Neural Network (Williams, 1998)** | $k_{\text{NN}}(\mathbf{x}, \mathbf{x}') = \frac{2}{\pi} \arcsin\left(\frac{2 \tilde{\mathbf{x}}^T \Sigma \tilde{\mathbf{x}}'}{\sqrt{(1+2\tilde{\mathbf{x}}^T\Sigma\tilde{\mathbf{x}})(1+2\tilde{\mathbf{x}}'^T\Sigma\tilde{\mathbf{x}}')}}\right)$<br>dengan $\tilde{\mathbf{x}} = [1, \mathbf{x}^T]^T$ | - $\Sigma = \operatorname{diag}(\sigma_0^2, \sigma_v^2)$ | - $\sigma_0^2$: Variansi bias neuron.<br>- $\sigma_v^2$: Variansi bobot input neuron. Menghasilkan fungsi dengan karakteristik transisi sigmoid non-stasioner. |

---

## 2. Analisis Perilaku Matematis & Geometris Variasi Hyperparameter

Ketika kita memvariasikan hyperparameter pada fungsi kernel, bentuk kovariansi dan realisasi sampel fungsi mengalami perubahan geometris yang khas:

```
[ VARIABEL INPUT x ] ───▶ [ KERNEL k(x, x') DENGAN HYPERPARAMETER ] ───▶ [ POLA SAMPEL FUNGSI f(x) ]
                                      │
            ┌─────────────────────────┼─────────────────────────┐
            ▼                         ▼                         ▼
   [ LENGTHSCALE l ]        [ SIGNAL VARIANCE sigma_f^2 ]    [ SHAPE / ORDER (alpha, nu, p) ]
   - l kecil: Osilasi cepat   - sigma_f besar: Amplitudo luas - alpha kecil: Fluktuasi multi-skala
   - l besar: Kaku / datar    - sigma_f kecil: Rapat ke nol   - p: Panjang gelombang periodik
```

### 1. Pengaruh *Lengthscale* ($l$):
* **Nilai $l$ Kecil (misal $l = 0.2$)**:
  * Nilai $k(r) = \exp(-r^2 / 2l^2)$ meluruh sangat curam mendekati nol meskipun jarak $r$ sangat dekat.
  * Dua titik yang sedikit berjauhan sudah tidak saling berkorelasi ($\operatorname{cov} \approx 0$).
  * **Sampel Fungsi**: Berfluktuasi liar, banyak puncak dan lembah dalam rentang sempit (frekuensi spasial tinggi).
* **Nilai $l$ Besar (misal $l = 3.0$)**:
  * Nilai $k(r)$ bertahan mendekati $1.0$ hingga jarak $r$ yang relatif jauh.
  * Titik-titik yang berjauhan masih dipaksa memiliki nilai yang mirip.
  * **Sampel Fungsi**: Berubah sangat lambat, tampak seperti garis lurus atau lengkungan landai (kaku).

---

### 2. Pengaruh *Signal Variance* ($\sigma_f^2$):
* Mengalikan seluruh matriks kovariansi dengan skalar $\sigma_f^2$.
* Sesuai sifat aljabar: Jika $f \sim \mathcal{GP}(0, k)$, maka $\operatorname{Var}[f(x)] = k(x, x) = \sigma_f^2$.
* **Nilai $\sigma_f^2$ Besar**: Simpangan fungsi terhadap sumbu nol sangat lebar (pita ketidakpastian prior $\pm 2\sigma_f$ melebar).
* **Nilai $\sigma_f^2$ Kecil**: Seluruh lintasan fungsi terkunci rapat di sekitar mean $\mu(x)=0$.

---

### 3. Pengaruh Parameter Bentuk (*Shape Parameters*):
* **Parameter $\alpha$ pada Rational Quadratic ($k_{\text{RQ}}$)**:
  * $k_{\text{RQ}}$ adalah campuran dari kernel RBF dengan berbagai *lengthscale*: $k_{\text{RQ}}(r) = \int p(l^{-2}) \exp(-r^2 / 2l^2) \, d(l^{-2})$.
  * **Nilai $\alpha$ Kecil (misal $\alpha = 0.2$)**: Bobot campuran merata pada rentang skala yang sangat lebar. Sampel fungsi menampilkan fluktuasi cepat (skala kecil) yang menumpang di atas tren gelombang panjang (skala besar).
  * **Nilai $\alpha$ Besar (misal $\alpha \ge 10$)**: Distribusi mengkerut ke satu skala tunggal, berperilaku persis seperti Squared Exponential biasa.
* **Parameter Periode ($p$) dan $l$ pada Periodic ($k_{\text{Per}}$)**:
  * **$p$**: Mengatur jarak spasial antara satu puncak ke puncak berikutnya ($f(x + p) \approx f(x)$).
  * **$l$ di dalam fungsi sinus**: Mengatur seberapa kompleks variasi bentuk gelombang di dalam satu siklus $p$. Jika $l$ kecil, gelombang memiliki harmonik/puncak tajam; jika $l$ besar, gelombang menjadi sinusoidal murni yang lembut.
* **Parameter Titik Tumpu ($c$) pada Linear Kernel ($k_{\text{Lin}}$)**:
  * Variansi prior adalah $\operatorname{Var}[f(x)] = \sigma_b^2 + \sigma_v^2 (x - c)^2$.
  * Pada titik $x = c$, ketidakpastian bernilai minimum ($\sigma_b^2$). Semakin jauh $x$ bergerak ke kiri ($x < c$) atau ke kanan ($x > c$), variansi bertambah secara kuadratik, sehingga sampel fungsi berupa garis lurus yang menyebar berbentuk corong (*fan shape*) bertumpu di titik $c$.

---

## 3. Matriks Desain Eksperimen Variasi Hyperparameter

Berikut daftar pengujian komputasi yang siap divisualisasikan dalam bentuk grafik:

| No | Nama Eksperimen | Kernel yang Diuji | Nilai Parameter yang Divariasikan | Visualisasi yang Dihasilkan |
|---|---|---|---|---|
| **Exp 1** | **Variasi Lengthscale $l$** | Squared Exponential ($k_{\text{SE}}$) | $l \in \{0.2, 1.0, 3.0\}$, fixed $\sigma_f=1.0$ | 3 panel perbandingan: kurva $k(r)$ dan 3 kurva sampel fungsi $f(x)$ |
| **Exp 2** | **Variasi Signal Variance $\sigma_f^2$** | Squared Exponential ($k_{\text{SE}}$) | $\sigma_f^2 \in \{0.25, 1.0, 4.0\}$, fixed $l=1.0$ | 3 panel variasi amplitudo vertikal terhadap rentang $[-4, 4]$ |
| **Exp 3** | **Variasi Parameter $\nu$ Matérn** | Matérn ($\nu=1/2, 3/2, 5/2, \infty$) | $\nu \in \{0.5, 1.5, 2.5, 10.0\}$, fixed $l=1.0, \sigma_f=1.0$ | Memperlihatkan profil kernel meluruh vs tingkat kekasaran sampel |
| **Exp 4** | **Variasi Multi-skala $\alpha$ RQ** | Rational Quadratic ($k_{\text{RQ}}$) | $\alpha \in \{0.1, 0.5, 2.0, 20.0\}$, fixed $l=1.0, \sigma_f=1.0$ | Menunjukkan transisi dari multi-scale mixture menuju single-scale SE |
| **Exp 5** | **Variasi Power $\gamma$ pada $\gamma$-Exp** | $\gamma$-Exponential ($k_\gamma$) | $\gamma \in \{0.5, 1.0, 1.5, 2.0\}$, fixed $l=1.0, \sigma_f=1.0$ | Bentuk puncak pada $r=0$ dan pengaruhnya pada sampel |
| **Exp 6** | **Variasi Periode $p$ & $l$ Periodic** | Exp-Sine-Squared ($k_{\text{Per}}$) | $p \in \{1.0, 2.5, 5.0\}$ dan $l_{\text{per}} \in \{0.5, 1.5\}$ | Pengulangan siklus beraturan dan harmonik gelombang |
| **Exp 7** | **Variasi Offset $c$ & Slope $\sigma_v^2$ Linear** | Linear Kernel ($k_{\text{Lin}}$) | $c \in \{-2.0, 0.0, 2.0\}$ dan $\sigma_v^2 \in \{0.5, 2.0\}$ | Corong ketidakpastian non-stasioner yang bertumpu di titik $c$ |
| **Exp 8** | **Variasi Bobot $\Sigma$ Neural Network** | Arc-sine Kernel ($k_{\text{NN}}$) | $\sigma_0^2 \in \{0.1, 1.0, 5.0\}$ dan $\sigma_v^2 \in \{1.0, 10.0\}$ | Sampel fungsi mirip aktivasi neural network non-stasioner |

---

## 4. Rekomendasi Struktur Baru Section 1 pada `kredit_bimbingan_03.tex`

Dengan memfokuskan Section 1 murni pada variasi hyperparameter seluruh kernel Rasmussen, struktur bab menjadi sangat rapi dan padat:

* **\section{Karakteristik Fungsi Kovariansi (Kernel) dan Eksplorasi Hyperparameter}**
  * **\subsection{Definisi Formal dan Klasifikasi Sifat Kernel}**
    * Syarat *Positive Semi-Definite* (PSD).
    * Pembagian kelas: Stasioner (invarian translasi) vs Non-Stasioner (tergantung lokasi absolut).
  * **\subsection{Katalog Fungsi Kovariansi Buku Teks (Rasmussen \& Williams)}**
    * Formula analitik lengkap: Squared Exponential, Matérn, Rational Quadratic, $\gamma$-Exponential, Periodic, Cosine, Linear, Polynomial, Neural Network, dan White Noise.
  * **\subsection{Analisis Efek Fisis Hyperparameter Ruang Fungsi}**
    * Peran *lengthscale* ($l$), *signal variance* ($\sigma_f^2$), parameter bentuk ($\alpha, \gamma, \nu$), periode ($p$), dan parameter non-stasioner ($c, \sigma_v^2$).
  * **\subsection{Hasil Eksperimen Komputasi & Visualisasi Variasi Hyperparameter}**
    * Grafik komparasi profil kovariansi $k(r)$ atau $k(x, x')$.
    * Grafik galeri realisasi sampel fungsi prior $\mathbf{f} \sim \mathcal{GP}(0, K)$ untuk tiap skenario variasi parameter.